# YOLO11n-Ghost-Hybrid-P3P4-Medium Model Summary

**Team:** P P Satya Karthikeya · B Karthikeya · M Karthik Reddy · P Rohit

Comprehensive analysis and visualization of the ultra-lightweight insulator defect detection model.

## 1. Import Required Libraries

Import necessary libraries for loading and analyzing the YOLO model.

In [1]:
import sys
import torch
from ultralytics import YOLO
import torch.nn as nn

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU Memory: 3.94 GB


## 2. Load the Trained Model

Load the best trained model from exp_012 (head-only fine-tune @ 768px).

In [2]:
model_path = "experiments/exp_012_head_finetune_768/weights/best.pt"
model = YOLO(model_path)

print(f"✅ Model loaded: {model_path}")
print(f"Model name: {model.model.model_name if hasattr(model.model, 'model_name') else 'YOLO11n-Ghost-Hybrid'}")

✅ Model loaded: experiments/exp_012_head_finetune_768/weights/best.pt
Model name: YOLO11n-Ghost-Hybrid


## 3. Display Model Summary

Show detailed model architecture, layer information, and parameter counts.

In [3]:
# Get the underlying PyTorch model
pytorch_model = model.model

# Display model info
print("=" * 80)
print("MODEL SUMMARY - YOLO11n-Ghost-Hybrid-P3P4-Medium")
print("=" * 80)
pytorch_model.info()

MODEL SUMMARY - YOLO11n-Ghost-Hybrid-P3P4-Medium
YOLO11n-ghost-hybrid-p3p4-medium summary: 210 layers, 867,664 parameters, 0 gradients, 5.8 GFLOPs


(210, 867664, 0, 5.8352832)

## 4. Analyze Model Architecture & Parameters

Detailed breakdown of model components and parameter distribution.

In [4]:
# Count total parameters
total_params = sum(p.numel() for p in pytorch_model.parameters())
trainable_params = sum(p.numel() for p in pytorch_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print("\n" + "=" * 80)
print("PARAMETER ANALYSIS")
print("=" * 80)
print(f"Total Parameters:     {total_params:>12,}  ({total_params/1e6:>6.2f}M)")
print(f"Trainable Parameters: {trainable_params:>12,}  ({trainable_params/1e6:>6.2f}M)")
print(f"Frozen Parameters:    {frozen_params:>12,}  ({frozen_params/1e6:>6.2f}M)")
print(f"Model Size (FP32):    {(total_params * 4 / 1e6):>12.2f}  MB")

# Estimate FLOPs
print("\n" + "=" * 80)
print("ARCHITECTURE DETAILS")
print("=" * 80)
print(f"Input Resolution:     768 × 768 × 3")
print(f"Output Classes:       2 (Damaged_1, insulator)")
print(f"GFLOPs:               ~5.7 (estimated)")
print(f"\nBackbone:             GhostConv + C3Ghost (feature extraction)")
print(f"Head:                 DWConv-based FPN (depthwise separable)")
print(f"Detection Scales:     P3 (96×96), P4 (48×48)  [P5 removed]")

# Layer count by type
layer_types = {}
for module in pytorch_model.modules():
    layer_name = module.__class__.__name__
    if layer_name != 'Sequential' and layer_name != 'Model':
        layer_types[layer_name] = layer_types.get(layer_name, 0) + 1

print(f"\nLayer Composition:")
for layer_type, count in sorted(layer_types.items(), key=lambda x: x[1], reverse=True):
    if count > 0:
        print(f"  {layer_type:<20} ×{count:>3}")


PARAMETER ANALYSIS
Total Parameters:          867,664  (  0.87M)
Trainable Parameters:            0  (  0.00M)
Frozen Parameters:         867,664  (  0.87M)
Model Size (FP32):            3.47  MB

ARCHITECTURE DETAILS
Input Resolution:     768 × 768 × 3
Output Classes:       2 (Damaged_1, insulator)
GFLOPs:               ~5.7 (estimated)

Backbone:             GhostConv + C3Ghost (feature extraction)
Head:                 DWConv-based FPN (depthwise separable)
Detection Scales:     P3 (96×96), P4 (48×48)  [P5 removed]

Layer Composition:
  Conv2d               × 84
  BatchNorm2d          × 79
  Conv                 × 75
  Identity             × 40
  GhostConv            × 24
  GhostBottleneck      × 10
  C3Ghost              ×  4
  DWConv               ×  4
  Concat               ×  3
  Upsample             ×  2
  ModuleList           ×  2
  DetectionModel       ×  1
  SiLU                 ×  1
  SPPF                 ×  1
  MaxPool2d            ×  1
  Detect               ×  1
  DFL  

## 5. Model Performance & Validation Results

Accuracy metrics on the VOC insulator dataset.

In [5]:
print("\n" + "=" * 80)
print("VALIDATION RESULTS (VOC Dataset - 143 val images)")
print("=" * 80)

# Metrics from validated run
metrics = {
    "mAP50": "96.10%",
    "mAP50-95": "66.79%",
    "Precision": "93.65%",
    "Recall": "94.18%",
    "Damaged_1 AP50": "95.05%",
    "Insulator AP50": "97.15%"
}

for metric, value in metrics.items():
    print(f"{metric:<25} {value:>10}")

print("\n" + "=" * 80)
print("INFERENCE PERFORMANCE (Raspberry Pi 4/5, ONNX Runtime, FP32)")
print("=" * 80)
print(f"Inference only (median):  657.7 ms")
print(f"Full pipeline (median):   692.1 ms")
print(f"Throughput:               1.4 FPS")
print(f"Min latency:              643.9 ms")
print(f"Max latency:              697.1 ms")
print(f"Std deviation:            4.6 ms")


VALIDATION RESULTS (VOC Dataset - 143 val images)
mAP50                         96.10%
mAP50-95                      66.79%
Precision                     93.65%
Recall                        94.18%
Damaged_1 AP50                95.05%
Insulator AP50                97.15%

INFERENCE PERFORMANCE (Raspberry Pi 4/5, ONNX Runtime, FP32)
Inference only (median):  657.7 ms
Full pipeline (median):   692.1 ms
Throughput:               1.4 FPS
Min latency:              643.9 ms
Max latency:              697.1 ms
Std deviation:            4.6 ms


## 6. Comparison: Our Model vs Baselines

Compare the Ghost-Hybrid student against vanilla YOLO11n and teacher YOLO11s.

In [6]:
import pandas as pd

comparison_data = {
    "Model": ["Vanilla YOLO11n", "Ghost-Hybrid (Ours)", "Teacher YOLO11s"],
    "Parameters": ["2.58M", "867K", "9.43M"],
    "GFLOPs": ["—", "5.7", "21.3"],
    "Input (px)": ["640", "768", "640"],
    "mAP50": ["90.20%", "96.10%", "96.54%"],
    "mAP50-95": ["58.19%", "66.79%", "—"],
    "Precision": ["89.70%", "93.65%", "—"],
    "Recall": ["80.30%", "94.18%", "—"]
}

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "=" * 80)
print("MODEL COMPARISON")
print("=" * 80)
print(df_comparison.to_string(index=False))

print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)
print("✅ Ghost-Hybrid surpasses vanilla YOLO11n by  +5.90% mAP50")
print("✅ Uses 3.0× fewer parameters than vanilla YOLO11n")
print("✅ Matches YOLO11s teacher within 0.44% mAP50")
print("✅ Achieves teacher-level performance with 10.9× fewer parameters")


MODEL COMPARISON
              Model Parameters GFLOPs Input (px)  mAP50 mAP50-95 Precision Recall
    Vanilla YOLO11n      2.58M      —        640 90.20%   58.19%    89.70% 80.30%
Ghost-Hybrid (Ours)       867K    5.7        768 96.10%   66.79%    93.65% 94.18%
    Teacher YOLO11s      9.43M   21.3        640 96.54%        —         —      —

KEY INSIGHTS
✅ Ghost-Hybrid surpasses vanilla YOLO11n by  +5.90% mAP50
✅ Uses 3.0× fewer parameters than vanilla YOLO11n
✅ Matches YOLO11s teacher within 0.44% mAP50
✅ Achieves teacher-level performance with 10.9× fewer parameters


## 7. Model Architecture Diagram

Visualize the complete architecture with layer flow.

In [7]:
# Display the model string representation
print("\n" + "=" * 80)
print("MODEL STRUCTURE")
print("=" * 80)
# Print structured model view
print(pytorch_model)


MODEL STRUCTURE
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): GhostConv(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), groups=32, bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
    )
    (2): C3Ghost(
      (cv1): Conv(
        (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03,

## 8. Summary & Key Takeaways

**Architecture:** YOLO11n-Ghost-Hybrid-P3P4-Medium  
**Total Parameters:** 867K (0.867M)  
**GFLOPs:** 5.7  
**Validation mAP50:** 96.10%  
**Inference (Raspberry Pi):** 1.4 FPS  

### Why This Design Works

1. **GhostConv Backbone** — ~50% fewer parameters through cheap linear operations without accuracy loss
2. **C3Ghost Blocks** — Cross-stage partial design with efficient bottlenecks, more capacity at detection scales (P3/P4)
3. **DWConv Head** — ~9× lighter than standard convolution heads, ideal for edge deployment
4. **P3+P4 detection only** — Removes P5 (for large objects), focusing all capacity on small/medium insulator defects
5. **768px input** — 44% more pixels than baseline 640px, capturing subtle defect details within receptive field limits

### Comparison Summary

| Metric | Vanilla YOLO11n | Ghost-Hybrid (Ours) | Improvement |
|---|---|---|---|
| **mAP50** | 90.20% | 96.10% | **+5.90%** |
| **Parameters** | 2.58M | 867K | **3.0× fewer** |
| **vs Teacher (YOLO11s)** | — | Within 0.44% mAP50 | **99.5% efficiency** |

### Deployment Targets

- **Raspberry Pi 4/5** — ONNX Runtime, CPU inference, ~1.4 FPS
- **NVIDIA Jetson Orin Nano** — TensorRT FP16 export
- **Edge TPU / Mobile** — TFLite INT8 export

---

*See full technical details in:*
- *`docs/ARCHITECTURE.md` — detailed layer-by-layer breakdown*
- *`BENCHMARK.md` — complete accuracy, speed, and comparison benchmarks*